# Bab 18: Ketidakpastian, Kalibrasi, dan Keputusan

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab18/data_bansos.py": "\"\"\"Rumah tangga sintetis untuk proxy means test (PMT) bantuan sosial (ilustrasi).\n\nSetiap rumah tangga punya ciri yang mudah diamati petugas (aset, kondisi\nrumah, pendidikan kepala rumah tangga) dan pengeluaran per kapita yang tidak\ndiamati langsung. Wilayah 3T (terdepan, terluar, tertinggal) sengaja dibuat\nlebih sedikit, lebih miskin, dan dengan hubungan aset--pengeluaran yang lebih\nlemah. Angka TIDAK mencerminkan data DTKS/Regsosek yang sebenarnya.\n\"\"\"\nimport numpy as np\n\nWILAYAH = [\"perkotaan\", \"perdesaan\", \"3T\"]\nGARIS = np.log(600)            # garis kemiskinan: Rp600 ribu/kapita/bulan\n\ndef buat_data(n=8000, seed=0):\n    rng = np.random.default_rng(seed)\n    w = rng.choice(3, n, p=[0.5, 0.4, 0.1])\n    art = rng.integers(1, 9, n)                                   # anggota rumah tangga\n    didik = np.clip(rng.normal(np.array([11, 8, 6])[w], 3), 0, 16)      # tahun sekolah KRT\n    laten = rng.normal(0, 1, n)                                   # kesejahteraan tersembunyi\n    lantai_tanah = (rng.normal(-laten - np.array([1.2, 0.6, -0.2])[w], 1) > 0).astype(float)\n    listrik = (rng.normal(laten + np.array([2.5, 1.5, 0.0])[w], 1) > 0).astype(float)\n    motor = (rng.normal(laten + np.array([0.6, 0.6, -0.4])[w], 1) > 0).astype(float)\n    kulkas = (rng.normal(laten + np.array([0.3, -0.3, -1.0])[w], 1) > 0).astype(float)\n    luas = np.exp(rng.normal(2.6 + 0.3 * laten - 0.08 * art, 0.3))  # m2 per kapita\n    # pengeluaran: aset informatif di kota, kurang informatif di 3T\n    kuat = np.array([0.35, 0.28, 0.15])[w]\n    log_c = (np.array([7.05, 6.80, 6.60])[w] + kuat * laten + 0.025 * didik\n             - 0.05 * art + rng.normal(0, np.array([0.18, 0.22, 0.40])[w]))\n    X = np.c_[np.eye(3)[w], art, didik, lantai_tanah, listrik, motor, kulkas, np.log(luas)]\n    return X, log_c, w\n",
"bab18/hujan_ensemble.py": "import numpy as np\nimport torch\nfrom torch import nn\n\n# Data yang sama dengan Bab 1: 15 pengamatan, alat rusak Juni-Agustus.\nrng = np.random.default_rng(3)\nf = lambda x: 200 + 150 * np.cos(2 * np.pi * x)\nx = rng.uniform(0, 1, 40)\nx = x[(x < 5/12) | (x > 8/12)][:15]\nt = f(x) + rng.normal(0, 40, x.size)\nX = torch.tensor((x[:, None] - 0.5) * 4, dtype=torch.float32)       # skala masukan\nT = torch.tensor(((t - 200) / 100)[:, None], dtype=torch.float32)   # skala target\nmm = lambda a: a * 100 + 200                                         # kembali ke mm\n\ndef latih(idx, p_drop=0.0, seed=0, iterasi=1500):\n    torch.manual_seed(seed)\n    net = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Dropout(p_drop),\n                        nn.Linear(64, 64), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(64, 1))\n    opt = torch.optim.Adam(net.parameters(), 3e-3, weight_decay=1e-3)   # ~ prior Gaussian\n    for _ in range(iterasi):\n        opt.zero_grad(); ((net(X[idx]) - T[idx]) ** 2).mean().backward(); opt.step()\n    return net\n\nsemua = np.arange(15)\ntunggal = latih(semua)                                    # satu jaringan biasa\nmc = latih(semua, p_drop=0.2)                             # MC dropout  #<1>\nboot = [rng.choice(15, 15) for _ in range(20)]            # ensemble + bootstrap  #<2>\nens = [latih(b, seed=s) for s, b in enumerate(boot)]\n\nxb = torch.tensor([[0.5 / 12], [3.5 / 12], [6.5 / 12], [9.5 / 12]])\nxb = (xb - 0.5) * 4\nwith torch.no_grad():\n    r1 = mm(tunggal(xb)[:, 0].numpy())\n    mc.train()                                            # dropout tetap aktif saat prediksi\n    S_mc = mm(torch.stack([mc(xb)[:, 0] for _ in range(500)]).numpy())\n    S_en = mm(torch.stack([n(xb)[:, 0] for n in ens]).numpy())\n    P = mm(torch.stack([n(X)[:, 0] for n in ens]).numpy())\noob = [np.mean([P[k, i] for k, b in enumerate(boot) if i not in b]) for i in range(15)]\ns_alea = np.sqrt(np.nanmean((np.array(oob) - t) ** 2))    # derau dari galat out-of-bag  #<3>\n\nprint(f\"taksiran derau aleatorik (out-of-bag): {s_alea:.0f} mm (sebenarnya 40 mm)\")\nprint(\"bulan    sebenarnya | tunggal | MC dropout | ensemble 20 jaringan\")\nprint(\"                    | rerata  | rerata sd  | rerata s_epi s_tot\")\nfor i, b in enumerate([\"Januari\", \"April\", \"Juli\", \"Oktober\"]):\n    s_tot = np.sqrt(S_en[:, i].var() + s_alea ** 2)       #<4>\n    print(f\"{b:8s} {f(1 / 24 + i / 4):9.0f}  | {r1[i]:6.0f}  | {S_mc[:, i].mean():6.0f}\"\n          f\" {S_mc[:, i].std():3.0f} | {S_en[:, i].mean():6.0f} {S_en[:, i].std():5.0f} {s_tot:5.0f}\")\n\nxg = np.linspace(0, 1, 121)                               # untuk gambar (tidak dicetak)\nwith torch.no_grad():\n    G = mm(torch.stack([n(torch.tensor((xg[:, None] - .5) * 4, dtype=torch.float32))[:, 0]\n                        for n in ens]).numpy())\nnp.savetxt(\"_kurva_hujan.txt\", np.c_[xg, f(xg), G.mean(0), G.std(0),\n           np.sqrt(G.var(0) + s_alea ** 2)], fmt=\"%.4f\")\nnp.savetxt(\"_titik_hujan.txt\", np.c_[x, t], fmt=\"%.4f\")\n",
"bab18/kalibrasi_bansos.py": "import numpy as np\nfrom scipy.optimize import minimize_scalar\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.isotonic import IsotonicRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom data_bansos import buat_data, GARIS\n\nX, log_c, w = buat_data()\ny = (log_c < GARIS).astype(int)                          # 1 = miskin\nX = StandardScaler().fit(X[:4000]).transform(X)\nL, K, U = slice(0, 4000), slice(4000, 6000), slice(6000, 8000)   # latih, kalibrasi, uji #<1>\n\nnet = MLPClassifier((128, 128), alpha=0, max_iter=400, random_state=0).fit(X[L], y[L])\nlogit = lambda p: np.log(p / (1 - p))\nz = logit(np.clip(net.predict_proba(X)[:, 1], 1e-7, 1 - 1e-7))\n\ndef nll(p, y):\n    p = np.clip(p, 1e-7, 1 - 1e-7)\n    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))\n\ndef ece(p, y, B=10):                                     # expected calibration error\n    b = np.minimum((p * B).astype(int), B - 1)\n    return sum(abs(p[b == k].mean() - y[b == k].mean()) * np.mean(b == k)\n               for k in range(B) if np.any(b == k))\n\nT = minimize_scalar(lambda T: nll(1 / (1 + np.exp(-z[K] / T)), y[K]),\n                    bounds=(0.05, 20), method=\"bounded\").x          # temperatur  #<2>\nplatt = LogisticRegression(C=1e6).fit(z[K, None], y[K])             #<3>\niso = IsotonicRegression(out_of_bounds=\"clip\").fit(z[K], y[K])      #<4>\nP = {\"tanpa kalibrasi\": 1 / (1 + np.exp(-z[U])),\n     f\"temperatur T={T:.2f}\": 1 / (1 + np.exp(-z[U] / T)),\n     \"Platt\": platt.predict_proba(z[U, None])[:, 1],\n     \"isotonik\": iso.predict(z[U])}\nprint(f\"miskin di data uji: {y[U].mean():.3f}\")\nprint(\"metode               akurasi    NLL    ECE  p>0,99 / p<0,01\")\nfor nama, p in P.items():\n    ekstrem = np.mean((p > 0.99) | (p < 0.01))\n    print(f\"{nama:20s} {np.mean((p > .5) == y[U]):7.3f} {nll(p, y[U]):6.3f}\"\n          f\" {ece(p, y[U]):6.3f} {ekstrem:10.2f}\")\nnp.savez(\"_kalibrasi.npz\", y=y[U], *P.values())\n",
"bab18/konformal_bansos.py": "import numpy as np\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nfrom data_bansos import buat_data, GARIS, WILAYAH\n\nX, y, w = buat_data()                                    # y = log pengeluaran per kapita\nL, K, U = slice(0, 4000), slice(4000, 6000), slice(6000, 8000)\nmodel = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,\n                                      random_state=0).fit(X[L], y[L])\nskor = np.abs(y[K] - model.predict(X[K]))                # skor ketidaksesuaian  #<1>\nyu, wu, mu = y[U], w[U], model.predict(X[U])\n\ndef kuantil(s, alfa=0.1):                                #<2>\n    n = len(s)\n    return np.quantile(s, np.ceil((n + 1) * (1 - alfa)) / n, method=\"higher\")\n\nq = kuantil(skor)\nq_w = np.array([kuantil(skor[w[K] == k]) for k in range(3)])    # Mondrian  #<3>\nprint(f\"target cakupan 90%; lebar selang satu kuantil: x{np.exp(2 * q):.2f}\")\nprint(\"               cakupan dengan kuantil ...\\n\"\n      \"wilayah       n   gabungan  per-wilayah   lebar\")\nfor k in range(3):\n    m = wu == k\n    c1 = np.mean(np.abs(yu[m] - mu[m]) <= q)\n    c2 = np.mean(np.abs(yu[m] - mu[m]) <= q_w[k])\n    print(f\"{WILAYAH[k]:10s} {m.sum():4d} {c1:10.3f} {c2:12.3f}   x{np.exp(2 * q_w[k]):.2f}\")\n\nlo, hi = mu - q_w[wu], mu + q_w[wu]                       # keputusan tiga arah  #<4>\nputus = np.where(hi < GARIS, \"layak\", np.where(lo > GARIS, \"tidak\", \"verif\"))\nmiskin = yu < GARIS\nprint(\"\\nwilayah    exclusion  inclusion | exclusion  inclusion  verifikasi\")\nprint(\"           (titik, ambang garis) | (konformal; verifikasi dikecualikan)\")\nfor k in range(3):\n    m = wu == k\n    ex1 = np.mean(mu[m & miskin] > GARIS)                 # miskin tetapi ditolak\n    in1 = np.mean(mu[m & ~miskin] < GARIS)                # tidak miskin tetapi diterima\n    ex2 = np.mean(putus[m & miskin] == \"tidak\")\n    in2 = np.mean(putus[m & ~miskin] == \"layak\")\n    print(f\"{WILAYAH[k]:10s} {ex1:8.3f} {in1:10.3f} | {ex2:8.3f} {in2:10.3f}\"\n          f\" {np.mean(putus[m] == 'verif'):10.3f}\")\nnp.savetxt(\"_konformal.txt\", np.c_[yu, mu, lo, hi, wu][:400], fmt=\"%.4f\")\n",
"bab18/ood_glif.py": "import sys, pathlib\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom sklearn.metrics import roc_auc_score\nsys.path.insert(0, str(pathlib.Path(__file__).parent.parent / \"bab16\"))\nfrom data_aksara import buat_data\n\nX, y = buat_data(8000)\nX = torch.tensor(X); y = torch.tensor(y)\nkenal = y < 8                                            # glif 8 dan 9 tidak pernah dilatih  #<1>\nXl, yl = X[:6000][kenal[:6000]], y[:6000][kenal[:6000]]\nXu_in, Xu_ood = X[6000:][kenal[6000:]], X[6000:][~kenal[6000:]]\ng = torch.Generator().manual_seed(1)                     # bercak acak\nacak = torch.rand(500, 400, generator=g) * (torch.rand(500, 400, generator=g) < 0.2)\n\ndef latih(seed):\n    torch.manual_seed(seed)\n    net = nn.Sequential(nn.Linear(400, 256), nn.ReLU(), nn.Linear(256, 8))\n    opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=1e-4)\n    for ep in range(15):\n        for idx in torch.randperm(len(yl)).split(64):\n            opt.zero_grad()\n            nn.functional.cross_entropy(net(Xl[idx]), yl[idx]).backward(); opt.step()\n    return net\n\nens = [latih(s) for s in range(5)]\nH = lambda p: -(p * torch.log(p + 1e-12)).sum(-1)       # entropi\ndef skor(Z):\n    with torch.no_grad():\n        P = torch.stack([n(Z).softmax(-1) for n in ens])  # (anggota, n, kelas)\n    pbar = P.mean(0)\n    return {\"1 - maks p (satu jaringan)\": 1 - P[0].max(-1).values,\n            \"entropi total (ensemble)\": H(pbar),                          #<2>\n            \"informasi mutual (epistemik)\": H(pbar) - H(P).mean(0)}, pbar\n\ns_in, p_in = skor(Xu_in)\nprint(f\"akurasi uji glif 0-7: {(p_in.argmax(-1) == y[6000:][kenal[6000:]]).float().mean():.3f}\")\nprint(\"skor                          AUROC glif 8-9  AUROC bercak acak\")\nfor nama in s_in:\n    a = [roc_auc_score(np.r_[np.zeros(len(s_in[nama])), np.ones(len(Z))],\n                       np.r_[s_in[nama].numpy(), skor(Z)[0][nama].numpy()])\n         for Z in (Xu_ood, acak)]\n    print(f\"{nama:30s} {a[0]:10.3f} {a[1]:14.3f}\")\ns_ood, p_ood = skor(Xu_ood)\nprint(f\"rerata maks p ensemble: kenal {p_in.max(-1).values.mean():.2f},\"\n      f\" glif baru {p_ood.max(-1).values.mean():.2f}\")\n",
"bab16/data_aksara.py": "\"\"\"Citra tulisan tangan sintetis 20 x 20 piksel untuk 10 kelas glif (ilustrasi).\n\nSetiap glif dibangun dari goresan sederhana (lengkung, lingkaran kecil, garis\ntegak dan mendatar) yang meniru gaya goresan aksara Jawa secara skematis.\nBentuknya TIDAK akurat sebagai aksara; untuk penelitian, gunakan himpunan data\ntulisan tangan aksara Jawa yang sebenarnya.\n\"\"\"\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\ndef _busur(cx, cy, r, a0, a1, n=12):\n    t = np.linspace(np.radians(a0), np.radians(a1), n)\n    return list(zip(cx + r * np.cos(t), cy + r * np.sin(t)))\n\ndef _glif():\n    G = []\n    G.append([_busur(.3, .5, .18, 180, 360) + [(.48, .8)], [(.48, .5), (.7, .5)], _busur(.7, .5, .15, 180, 360)])\n    G.append([[(.15, .7), (.15, .35)], _busur(.32, .35, .17, 180, 360), [(.49, .35), (.49, .7)], _busur(.66, .35, .17, 180, 360) + [(.83, .7)]])\n    G.append([_busur(.5, .4, .25, 180, 360), [(.25, .4), (.25, .75), (.75, .75), (.75, .4)], _busur(.5, .55, .07, 0, 360)])\n    G.append([[(.2, .3), (.8, .3)], [(.35, .3), (.35, .8)], [(.65, .3), (.65, .8)], _busur(.5, .8, .15, 0, 180)])\n    G.append([_busur(.35, .5, .2, 90, 450), [(.55, .5), (.85, .5)], [(.85, .25), (.85, .8)]])\n    G.append([_busur(.5, .35, .25, 180, 360), [(.25, .35), (.4, .8)], [(.75, .35), (.6, .8)], [(.4, .8), (.6, .8)]])\n    G.append([[(.2, .25), (.2, .75)], [(.2, .5), (.8, .5)], [(.8, .25), (.8, .75)], _busur(.5, .3, .1, 0, 360)])\n    G.append([_busur(.3, .4, .15, 0, 360), _busur(.7, .4, .15, 0, 360), [(.15, .75), (.85, .75)]])\n    G.append([[(.15, .3), (.35, .75), (.5, .4), (.65, .75), (.85, .3)]])\n    G.append([_busur(.5, .5, .3, 200, 520), [(.5, .2), (.5, .8)]])\n    return G\n\nGLIF = _glif()\n\ndef gambar(k, rng, ukuran=20):\n    S = 64; im = Image.new(\"L\", (S, S), 0); d = ImageDraw.Draw(im)\n    a = np.radians(rng.normal(0, 12)); sk = rng.normal(1, 0.12, 2); sh = rng.normal(0, 0.2)\n    M = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]]) @ np.array([[sk[0], sh], [0, sk[1]]])\n    geser = rng.normal(0, 0.06, 2)\n    for gores in GLIF[k]:\n        P = (np.array(gores) - 0.5 + rng.normal(0, 0.015, (len(gores), 2))) @ M.T + 0.5 + geser\n        d.line([tuple(p) for p in P * S], fill=255, width=int(rng.integers(3, 6)), joint=\"curve\")\n    x = np.asarray(im.resize((ukuran, ukuran), Image.BILINEAR), float) / 255\n    return np.clip(x + rng.normal(0, 0.1, x.shape), 0, 1)\n\ndef buat_data(n=6000, seed=0):\n    rng = np.random.default_rng(seed)\n    y = rng.integers(0, 10, n)\n    X = np.stack([gambar(k, rng) for k in y]).reshape(n, -1)\n    return X.astype(np.float32), y\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab18')
print('folder kerja:', os.getcwd())

## Listing 18.1: Satu jaringan, MC dropout, dan deep ensemble untuk data curah hujan

Berkas: `kode/bab18/hujan_ensemble.py`

In [ ]:
import numpy as np
import torch
from torch import nn

# Data yang sama dengan Bab 1: 15 pengamatan, alat rusak Juni-Agustus.
rng = np.random.default_rng(3)
f = lambda x: 200 + 150 * np.cos(2 * np.pi * x)
x = rng.uniform(0, 1, 40)
x = x[(x < 5/12) | (x > 8/12)][:15]
t = f(x) + rng.normal(0, 40, x.size)
X = torch.tensor((x[:, None] - 0.5) * 4, dtype=torch.float32)       # skala masukan
T = torch.tensor(((t - 200) / 100)[:, None], dtype=torch.float32)   # skala target
mm = lambda a: a * 100 + 200                                         # kembali ke mm

def latih(idx, p_drop=0.0, seed=0, iterasi=1500):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Dropout(p_drop),
                        nn.Linear(64, 64), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(64, 1))
    opt = torch.optim.Adam(net.parameters(), 3e-3, weight_decay=1e-3)   # ~ prior Gaussian
    for _ in range(iterasi):
        opt.zero_grad(); ((net(X[idx]) - T[idx]) ** 2).mean().backward(); opt.step()
    return net

semua = np.arange(15)
tunggal = latih(semua)                                    # satu jaringan biasa
mc = latih(semua, p_drop=0.2)                             # MC dropout  # (1)
boot = [rng.choice(15, 15) for _ in range(20)]            # ensemble + bootstrap  # (2)
ens = [latih(b, seed=s) for s, b in enumerate(boot)]

xb = torch.tensor([[0.5 / 12], [3.5 / 12], [6.5 / 12], [9.5 / 12]])
xb = (xb - 0.5) * 4
with torch.no_grad():
    r1 = mm(tunggal(xb)[:, 0].numpy())
    mc.train()                                            # dropout tetap aktif saat prediksi
    S_mc = mm(torch.stack([mc(xb)[:, 0] for _ in range(500)]).numpy())
    S_en = mm(torch.stack([n(xb)[:, 0] for n in ens]).numpy())
    P = mm(torch.stack([n(X)[:, 0] for n in ens]).numpy())
oob = [np.mean([P[k, i] for k, b in enumerate(boot) if i not in b]) for i in range(15)]
s_alea = np.sqrt(np.nanmean((np.array(oob) - t) ** 2))    # derau dari galat out-of-bag  # (3)

print(f"taksiran derau aleatorik (out-of-bag): {s_alea:.0f} mm (sebenarnya 40 mm)")
print("bulan    sebenarnya | tunggal | MC dropout | ensemble 20 jaringan")
print("                    | rerata  | rerata sd  | rerata s_epi s_tot")
for i, b in enumerate(["Januari", "April", "Juli", "Oktober"]):
    s_tot = np.sqrt(S_en[:, i].var() + s_alea ** 2)       # (4)
    print(f"{b:8s} {f(1 / 24 + i / 4):9.0f}  | {r1[i]:6.0f}  | {S_mc[:, i].mean():6.0f}"
          f" {S_mc[:, i].std():3.0f} | {S_en[:, i].mean():6.0f} {S_en[:, i].std():5.0f} {s_tot:5.0f}")

xg = np.linspace(0, 1, 121)                               # untuk gambar (tidak dicetak)
with torch.no_grad():
    G = mm(torch.stack([n(torch.tensor((xg[:, None] - .5) * 4, dtype=torch.float32))[:, 0]
                        for n in ens]).numpy())
np.savetxt("_kurva_hujan.txt", np.c_[xg, f(xg), G.mean(0), G.std(0),
           np.sqrt(G.var(0) + s_alea ** 2)], fmt="%.4f")
np.savetxt("_titik_hujan.txt", np.c_[x, t], fmt="%.4f")

## Listing 18.2: Kalibrasi pascapelatihan untuk klasifikasi rumah tangga miskin

Berkas: `kode/bab18/kalibrasi_bansos.py`

In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler
from data_bansos import buat_data, GARIS

X, log_c, w = buat_data()
y = (log_c < GARIS).astype(int)                          # 1 = miskin
X = StandardScaler().fit(X[:4000]).transform(X)
L, K, U = slice(0, 4000), slice(4000, 6000), slice(6000, 8000)   # latih, kalibrasi, uji # (1)

net = MLPClassifier((128, 128), alpha=0, max_iter=400, random_state=0).fit(X[L], y[L])
logit = lambda p: np.log(p / (1 - p))
z = logit(np.clip(net.predict_proba(X)[:, 1], 1e-7, 1 - 1e-7))

def nll(p, y):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def ece(p, y, B=10):                                     # expected calibration error
    b = np.minimum((p * B).astype(int), B - 1)
    return sum(abs(p[b == k].mean() - y[b == k].mean()) * np.mean(b == k)
               for k in range(B) if np.any(b == k))

T = minimize_scalar(lambda T: nll(1 / (1 + np.exp(-z[K] / T)), y[K]),
                    bounds=(0.05, 20), method="bounded").x          # temperatur  # (2)
platt = LogisticRegression(C=1e6).fit(z[K, None], y[K])             # (3)
iso = IsotonicRegression(out_of_bounds="clip").fit(z[K], y[K])      # (4)
P = {"tanpa kalibrasi": 1 / (1 + np.exp(-z[U])),
     f"temperatur T={T:.2f}": 1 / (1 + np.exp(-z[U] / T)),
     "Platt": platt.predict_proba(z[U, None])[:, 1],
     "isotonik": iso.predict(z[U])}
print(f"miskin di data uji: {y[U].mean():.3f}")
print("metode               akurasi    NLL    ECE  p>0,99 / p<0,01")
for nama, p in P.items():
    ekstrem = np.mean((p > 0.99) | (p < 0.01))
    print(f"{nama:20s} {np.mean((p > .5) == y[U]):7.3f} {nll(p, y[U]):6.3f}"
          f" {ece(p, y[U]):6.3f} {ekstrem:10.2f}")
np.savez("_kalibrasi.npz", y=y[U], *P.values())

## Listing 18.3: Deteksi masukan asing dengan ensemble lima jaringan

Berkas: `kode/bab18/ood_glif.py`

In [ ]:
import sys, pathlib
import numpy as np
import torch
from torch import nn
from sklearn.metrics import roc_auc_score
sys.path.insert(0, str(pathlib.Path.cwd().parent / "bab16"))
from data_aksara import buat_data

X, y = buat_data(8000)
X = torch.tensor(X); y = torch.tensor(y)
kenal = y < 8                                            # glif 8 dan 9 tidak pernah dilatih  # (1)
Xl, yl = X[:6000][kenal[:6000]], y[:6000][kenal[:6000]]
Xu_in, Xu_ood = X[6000:][kenal[6000:]], X[6000:][~kenal[6000:]]
g = torch.Generator().manual_seed(1)                     # bercak acak
acak = torch.rand(500, 400, generator=g) * (torch.rand(500, 400, generator=g) < 0.2)

def latih(seed):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(400, 256), nn.ReLU(), nn.Linear(256, 8))
    opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=1e-4)
    for ep in range(15):
        for idx in torch.randperm(len(yl)).split(64):
            opt.zero_grad()
            nn.functional.cross_entropy(net(Xl[idx]), yl[idx]).backward(); opt.step()
    return net

ens = [latih(s) for s in range(5)]
H = lambda p: -(p * torch.log(p + 1e-12)).sum(-1)       # entropi
def skor(Z):
    with torch.no_grad():
        P = torch.stack([n(Z).softmax(-1) for n in ens])  # (anggota, n, kelas)
    pbar = P.mean(0)
    return {"1 - maks p (satu jaringan)": 1 - P[0].max(-1).values,
            "entropi total (ensemble)": H(pbar),                          # (2)
            "informasi mutual (epistemik)": H(pbar) - H(P).mean(0)}, pbar

s_in, p_in = skor(Xu_in)
print(f"akurasi uji glif 0-7: {(p_in.argmax(-1) == y[6000:][kenal[6000:]]).float().mean():.3f}")
print("skor                          AUROC glif 8-9  AUROC bercak acak")
for nama in s_in:
    a = [roc_auc_score(np.r_[np.zeros(len(s_in[nama])), np.ones(len(Z))],
                       np.r_[s_in[nama].numpy(), skor(Z)[0][nama].numpy()])
         for Z in (Xu_ood, acak)]
    print(f"{nama:30s} {a[0]:10.3f} {a[1]:14.3f}")
s_ood, p_ood = skor(Xu_ood)
print(f"rerata maks p ensemble: kenal {p_in.max(-1).values.mean():.2f},"
      f" glif baru {p_ood.max(-1).values.mean():.2f}")

## Listing 18.4: Selang konformal untuk pengeluaran rumah tangga dan keputusan tiga arah

Berkas: `kode/bab18/konformal_bansos.py`

In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from data_bansos import buat_data, GARIS, WILAYAH

X, y, w = buat_data()                                    # y = log pengeluaran per kapita
L, K, U = slice(0, 4000), slice(4000, 6000), slice(6000, 8000)
model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                      random_state=0).fit(X[L], y[L])
skor = np.abs(y[K] - model.predict(X[K]))                # skor ketidaksesuaian  # (1)
yu, wu, mu = y[U], w[U], model.predict(X[U])

def kuantil(s, alfa=0.1):                                # (2)
    n = len(s)
    return np.quantile(s, np.ceil((n + 1) * (1 - alfa)) / n, method="higher")

q = kuantil(skor)
q_w = np.array([kuantil(skor[w[K] == k]) for k in range(3)])    # Mondrian  # (3)
print(f"target cakupan 90%; lebar selang satu kuantil: x{np.exp(2 * q):.2f}")
print("               cakupan dengan kuantil ...\n"
      "wilayah       n   gabungan  per-wilayah   lebar")
for k in range(3):
    m = wu == k
    c1 = np.mean(np.abs(yu[m] - mu[m]) <= q)
    c2 = np.mean(np.abs(yu[m] - mu[m]) <= q_w[k])
    print(f"{WILAYAH[k]:10s} {m.sum():4d} {c1:10.3f} {c2:12.3f}   x{np.exp(2 * q_w[k]):.2f}")

lo, hi = mu - q_w[wu], mu + q_w[wu]                       # keputusan tiga arah  # (4)
putus = np.where(hi < GARIS, "layak", np.where(lo > GARIS, "tidak", "verif"))
miskin = yu < GARIS
print("\nwilayah    exclusion  inclusion | exclusion  inclusion  verifikasi")
print("           (titik, ambang garis) | (konformal; verifikasi dikecualikan)")
for k in range(3):
    m = wu == k
    ex1 = np.mean(mu[m & miskin] > GARIS)                 # miskin tetapi ditolak
    in1 = np.mean(mu[m & ~miskin] < GARIS)                # tidak miskin tetapi diterima
    ex2 = np.mean(putus[m & miskin] == "tidak")
    in2 = np.mean(putus[m & ~miskin] == "layak")
    print(f"{WILAYAH[k]:10s} {ex1:8.3f} {in1:10.3f} | {ex2:8.3f} {in2:10.3f}"
          f" {np.mean(putus[m] == 'verif'):10.3f}")
np.savetxt("_konformal.txt", np.c_[yu, mu, lo, hi, wu][:400], fmt="%.4f")